---

# Feature Store : Time-Windowed Features and the new Aggregate Feature Views
___
# 
## Time-Windowed Features for ML: The Sparse Data Challenge


### The Core Problem: Time Drift with Sparse Events

When building ML features from event data, we often need **time-windowed aggregations** — for example, "count of purchases in the last 7 days" or "sum of revenue in the last 30 days". These features must be computed **as-of the prediction timestamp** (the "spine" timestamp) to avoid data leakage and ensure training/inference consistency.

The challenge arises when event data is **sparse** — entities don't have events every day. Consider a typical pattern where a subscriber might have events on only 20% of days:

```
Entity A's events:     Jan 1  ...  Jan 5  ...  Jan 10  ...  Jan 15
                         ↓          ↓           ↓            ↓
                       event      event       event        event

Prediction timestamps: Jan 1, Jan 2, Jan 3, Jan 4, Jan 5, Jan 6, ... Jan 15
                         ↓      ↓      ↓      ↓      ↓      ↓           ↓
                       needs  needs  needs  needs  needs  needs      needs
                       features for EVERY prediction date
```


### The Naive Approach: Pre-Computed Event-Relative Features

A common pattern is to pre-compute windowed features **at each event timestamp** and store them in a feature table:

```
Event-relative features (sparse — only exists on event days):
| ENTITY | EVENT_DATE | COUNT_7D | SUM_REVENUE_7D |
|--------|------------|----------|----------------|
| A      | Jan 5      | 15       | $500           |
| A      | Jan 10     | 20       | $700           |
```

Then use a **point-in-time (ASOF) join** to retrieve the "most recent" feature row for each prediction:

```sql
SELECT * FROM predictions p
ASOF JOIN features f 
  ON p.entity = f.entity 
  AND p.prediction_date >= f.event_date
```

**This creates a fundamental problem: the window is anchored to the wrong timestamp.**

For a prediction on **Jan 8**, the ASOF join retrieves features from **Jan 5** (the most recent event). But the Jan 5 feature row contains a 7-day window computed relative to Jan 5:

```
Jan 5 feature row (7-day window, excluding current day):
Window: Dec 29 → Jan 4

What Jan 8 prediction SHOULD have (7-day window, excluding current day):
Window: Jan 1 → Jan 7
```

The windows don't just differ in recency — they're **shifted by 3 days** in this example:

| Window Segment | Jan 5 Feature | Jan 8 Expected | Problem |
|----------------|---------------|----------------|---------|
| Dec 29-31 | ✅ Included | ❌ Should NOT be | **Erroneously included** |
| Jan 1-4 | ✅ Included | ✅ Should be | Correct overlap |
| Jan 5-7 | ❌ Not included | ✅ Should be | **Erroneously excluded** |

The feature window is anchored to the wrong timestamp, causing:
- **Window shift**: Events from Dec 29-31 are wrongly included; events from Jan 5-7 are wrongly excluded
- **Magnitude depends on gap**: The longer the gap between event and prediction, the worse the drift
- **Inconsistent semantics**: A "7-day window" covers different calendar periods for different predictions
- **Training/inference skew**: Model learns from incorrectly-shifted windows during training


### The Expensive Solution: Dense Pre-Computation

One solution is to pre-compute features for **every possible prediction date** (dense representation):

```
Dense features (one row per entity per day):
| ENTITY | AS_OF_DATE | COUNT_7D | SUM_REVENUE_7D |
|--------|------------|----------|----------------|
| A      | Jan 1      | 5        | $200           |
| A      | Jan 2      | 5        | $200           |  ← Same as Jan 1 (no new events)
| A      | Jan 3      | 5        | $200           |  ← Same as Jan 1
| A      | Jan 4      | 5        | $200           |  ← Same as Jan 1
| A      | Jan 5      | 15       | $500           |  ← Updated with Jan 5 event
| ...    | ...        | ...      | ...            |
```

This eliminates time drift — every prediction date has correctly-anchored features and also enables a more efficient inner-join at retrieval time. But the cost could be prohibitive:
```
| Approach                | Rows (10M entities × 365 days)   | Storage   |
|-------------------------|----------------------------------|-----------|
| Sparse (20% event days) | ~730M rows                       |    ~60 GB |
| Dense (every day)       | ~3.65B rows                      |    ~2+ TB |
```
The dense approach requires **5x more storage** and proportionally more compute to maintain — impractical for large-scale feature stores. 

Snowflake provides several gap-filling features and interpolation functions for this purpose that can be used to perform the gap-filling either :
- incrementally on event-data arrival (e.g. via a Dynamic Table over the sparse data)
- at query time (e.g. via a View over the sparse data)


### The Efficient Solution: Tiled Pre-Computation + Query-Time Completion

The solution is to **partially pre-compute** aggregations at a coarse grain (tiles), then **complete the computation at query time** relative to the actual prediction timestamp.

**Step 1: Pre-compute daily tiles (sparse)**

Aggregate raw events to daily grain, storing only days with events:

```
Daily tiles (sparse — only days WITH events):
| ENTITY | TILE_START | COUNT_1D | SUM_REVENUE_1D |
|--------|------------|----------|----------------|
| A      | Jan 1      | 5        | $200           |
| A      | Jan 5      | 10       | $300           |
| A      | Jan 10     | 5        | $200           |
```

**Step 2: Compute final windows at query time from spine**

At query time, join tiles directly to the prediction spine and aggregate within the window:

```sql
SELECT 
    spine.entity,
    spine.prediction_date,
    SUM(tile.COUNT_1D) AS COUNT_7D,  -- Sum tiles within 7-day window
    SUM(tile.SUM_REVENUE_1D) AS SUM_REVENUE_7D
FROM prediction_spine spine
LEFT JOIN daily_tiles tile
    ON spine.entity = tile.entity
    AND tile.TILE_START BETWEEN spine.prediction_date - 7 
                           AND spine.prediction_date - 1
GROUP BY spine.entity, spine.prediction_date
```

This approach:
- **Correct semantics**: Window is anchored to `prediction_date`, not `event_date`
- **Sparse storage**: Only store tiles for days with events (~730M rows, not 3.65B)
- **Efficient query**: Aggregate a small number of tiles per entity (≤7 for a 7-day window)
- **No time drift**: Features are always correctly aligned to the prediction timestamp


### Trade-off Summary

| Approach | Storage | Correctness | Query Complexity |
|----------|---------|-------------|------------------|
| **Event-relative + ASOF** | ✅ Sparse | ❌ Time drift | ✅ Simple join |
| **Dense pre-computation** | ❌ 5x+ storage | ✅ Correct | ✅ Simple join |
| **Tiles + query-time** | ✅ Sparse | ✅ Correct | ⚠️ Aggregation at query |


The tiled approach achieves the best of both worlds: sparse storage with correct temporal semantics, at the cost of slightly more complex (but optimized) query-time aggregation.

---
## NOTEBOOK

This Notebook explores and demonstrates the creation and use of time-windowed aggregate features in Snowflake Feature Store in two sections.

__Section 1__  
Using the pre-computed event-relative approach over sparse and dense representations of the same event data, showing the difference in the result from the two.  

__Section 2__  
Explores the new Time Aggregation Feature API which provides a new declarative way to define time-windowed features using the new Feature class. This uses a similar approach to the efficient solution outlined above to derive temporally correct feature-values at retrieval time.
Instead of writing complex SQL with window functions, you specify what aggregations you want and the Feature Store handles the efficient computation of intermediate tiled results, and final point-in-time feature derivation at retrieval time.
This is new functionality is currently in limited private-preview.  If you want to execute the code examples in the second section of the Notebook you will need to install the latest Private Previe package which can be obtained, subject to requirements and approvals, from the Feature Store Product Management team.



#### Package Installations

##### ipywidgets

In [ ]:
!pip install -U ipywidgets

##### snowflake_ml_python : PrPr 
If you want to execute the code examples in the second section of the Notebook you will need to install the latest PrPr package as per below.  This can be obtained, subject to requirements, from the Feature Store Product Management team.

In [ ]:
# Install & verify updated pkg

import os
notebook_path = os.path.abspath(os.getcwd())
ws_path = os.path.dirname(notebook_path)
pkg_path = os.path.join(ws_path, "packages")

pkg_name = "snowflake_ml_python-1.23.0-py3-none-any.whl"

full_pkg_path = os.path.join(pkg_path, pkg_name)
full_pkg_path

!pip install -U "{full_pkg_path}"

In [ ]:
import importlib
import snowflake.ml.feature_store

# Force a deep reload of the feature store components
importlib.reload(snowflake.ml.feature_store)

# Explicitly import the new classes
from snowflake.ml.feature_store import  FeatureStore, Entity, FeatureView, Feature, CreationMode

print(f"Successfully loaded: {Feature}")

### General Imports and Session Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

# Snowflake Packages
from snowflake.snowpark import Session
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.version import VERSION
from snowflake.snowpark import functions as F, types as T

# # Snowflake Feature Store
# from snowflake.ml.feature_store import FeatureStore, 
from snowflake.ml.version import VERSION as SML_VERSION

print(f"Snowpark Version: {VERSION}")
print(f"Snowflake ML Version: {SML_VERSION}")

In [ ]:
# Initialize Snowflake Session
session = get_active_session()

# Modify to database you want to use for examples
db_name = 'TIME_WINDOWED_FVIEWS'

In [ ]:
session.sql(f'''CREATE DATABASE IF NOT EXISTS {db_name}''').collect()

## Section 1

### Source Data - ORDERS

In [ ]:
session.sql('''CREATE SCHEMA IF NOT EXISTS SOURCE_DATA''').collect()

A typical **event** table used as a source for time series aggregations will be sparse on the time-series.  The data will have gaps in time, where a product does not have an order every day.  

For demonstration purposes we will create a table with both a sparse and dense representation for similar order data. The order sales-amounts are the same for each product.

The order data for product 101 is a sparse time-series where we have gaps (no records) for the `[2023-01-02, 2023-01-03, 2023-01-05, 2023-01-06, 2023-01-07]` 

The order data for product 201 us a dense time-series we have a record per day for every product regardless of whether that had been a sale on that day.  i.e. we record zero (or null), for no-order days.

In [ ]:
sample_data = [
# Sparse time-series event data (gaps for PRODUCTKEY 101)
    ["2023-01-01", 101, 200],
    ["2023-01-04", 101, 250],
    ["2023-01-08", 101, 300], 

# Dense time-series event data (gap-filled for PRODUCTKEY 201) 
    ["2023-01-01", 201, 200],
    ["2023-01-02", 201, 0],
    ["2023-01-03", 201, 0],    
    ["2023-01-04", 201, 250],
    ["2023-01-05", 201, 0],
    ["2023-01-06", 201, 0],
    ["2023-01-07", 201, 0],
    ["2023-01-08", 201, 300],      
]

order_df = session.create_dataframe(sample_data).to_df(
    "ORDERDATE", "PRODUCTKEY", "SALESAMOUNT")

order_df = order_df.with_column("ORDERDATE", F.to_timestamp(order_df["ORDERDATE"]))

order_df.write.save_as_table(f"{db_name}.SOURCE_DATA.ORDERS", mode="overwrite")

order_df.show(15)

### Feature Store
Create a Feature Store.

In [ ]:
fs = FeatureStore(
    session=session,
    database=db_name,
    name="FEATURE_STORE",
    default_warehouse="SIMON_XS",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

In [ ]:
product_entity = Entity(name="PRODUCT", join_keys=["PRODUCTKEY"])
ent_product = fs.register_entity(product_entity)

### Feature Source Dataframe
We will use the Analytics Functions to create temporal features.  The Anlaytic Functions make it easier and more efficient to express time-lagged aggregations over source data, but we could also express these lagged features in full using Snowpark or SQL.

In [ ]:
order_tbl = session.table(f"{db_name}.SOURCE_DATA.ORDERS")

def custom_formatter(input_col, agg, window):
    return f"{agg}_{input_col}_{window.replace('-','LAG')}"
    
analytic_df = order_tbl.analytics.time_series_agg(
    time_col="ORDERDATE",
    group_by=["PRODUCTKEY"],
    aggs={"SALESAMOUNT": ["SUM"]},
    windows=["0D", "-1D", "-2D", "-3D", "-4D"],
    sliding_interval="1D", # The docs say this is deprecated.  You can include it, but changing the value seems to have no impact on the result!
                           # I would expect it to either fail if included, or modify the final result returned.  Not do Nothing?
    col_formatter=custom_formatter,
).drop("SALESAMOUNT")

analytic_df.sort(F.col('PRODUCTKEY'), F.col('ORDERDATE')).show(20)

In [ ]:
print(analytic_df.queries['queries'][0])

Looking at the result (and the generated SQL) we can see that the LAG values include the current-row value.  This is typically not what we are looking for. We want the feature-values to be up-to, but excluding the current time.  

If we include the current row, particularly if we are performing a timestamp truncation (e.g. timestamp to day), we could include partial data for the current time-grain which can create training skew in the model.

> __Note:__ Adjusting the `sliding_interval` argument (deprecated) doesn't seem to alter the result. e.g. if we alter it to 10D the result remains the same!  Therefore we have manually edited the SQL below to exclude the current window `CURRENT ROW` --> `INTERVAL '1 DAY' PRECEDING`


In [ ]:
analytic_sql_df = session.sql(f"""
SELECT 
    "PRODUCTKEY", 
    "ORDERDATE", 
    SALESAMOUNT AS "SALESAMOUNT",
    ZEROIFNULL(SUM("SALESAMOUNT") OVER (PARTITION BY "PRODUCTKEY"  ORDER BY "ORDERDATE" ASC NULLS FIRST  RANGE BETWEEN INTERVAL '1 DAY' PRECEDING  AND INTERVAL '1 DAY' PRECEDING)) AS "SUM_SALESAMOUNT_LAG1D", 
    ZEROIFNULL(SUM("SALESAMOUNT") OVER (PARTITION BY "PRODUCTKEY"  ORDER BY "ORDERDATE" ASC NULLS FIRST  RANGE BETWEEN INTERVAL '2 DAY' PRECEDING  AND INTERVAL '1 DAY' PRECEDING)) AS "SUM_SALESAMOUNT_LAG2D", 
    ZEROIFNULL(SUM("SALESAMOUNT") OVER (PARTITION BY "PRODUCTKEY"  ORDER BY "ORDERDATE" ASC NULLS FIRST  RANGE BETWEEN INTERVAL '3 DAY' PRECEDING  AND INTERVAL '1 DAY' PRECEDING)) AS "SUM_SALESAMOUNT_LAG3D", 
    ZEROIFNULL(SUM("SALESAMOUNT") OVER (PARTITION BY "PRODUCTKEY"  ORDER BY "ORDERDATE" ASC NULLS FIRST  RANGE BETWEEN INTERVAL '4 DAY' PRECEDING  AND INTERVAL '1 DAY' PRECEDING)) AS "SUM_SALESAMOUNT_LAG4D"
 FROM {db_name}.SOURCE_DATA.ORDERS
""")

analytic_sql_df.sort(F.col('PRODUCTKEY'), F.col('ORDERDATE')).show(20)

We use the dataframe SQL derived from the analytics-functions as the source for our standard Feature Store Feature View.

In [ ]:
#fs.delete_feature_view('FV_PRODUCT_SALES_LAG_FEATURES','V1')

fv_product_sales_lag_features = FeatureView(name="FV_PRODUCT_SALES_LAG_FEATURES", 
                               entities=[ent_product], 
                               feature_df=analytic_sql_df,
                               timestamp_col="ORDERDATE",
                               refresh_freq="1days"
                               )
fv_product_sales_lag_features_v1 = fs.register_feature_view(fv_product_sales_lag_features, "V1", overwrite=True)

### Retrieve Training Data

We need to create a Spine with the products and dates that we wish to retrieve our training data for.  It's good practice to store Spines in a dedicated schema.

In [ ]:
session.sql('''CREATE SCHEMA IF NOT EXISTS SPINES''').collect()

We now create a Spine with a records covering the full date-range of the `ORDER` data and Feature View containing the aggregate time-lagged features.

In [ ]:
spine_dates = [
    ["2023-01-01"],
    ["2023-01-02"],
    ["2023-01-03"],
    ["2023-01-04"],
    ["2023-01-05"],
    ["2023-01-06"],    
    ["2023-01-07"],    
    ["2023-01-08"],    
]
spine_dates_df = session.create_dataframe(spine_dates).to_df( "SPINEDATE").with_column("SPINEDATE", F.to_timestamp(F.col("SPINEDATE")))

spine_products = [101,201]
spine_products_df = session.create_dataframe(spine_products).to_df( "PRODUCTKEY")

# Cartesian Join to get all combinations of Product and Spine Date
spine_df = spine_products_df.join(spine_dates_df)

spine_df.write.save_as_table(f"{db_name}.SPINES.SPINE", mode="overwrite")

spine_tbl = session.table(f"{db_name}.SPINES.SPINE")

spine_tbl.sort(F.col('PRODUCTKEY'), F.col('SPINEDATE')).show(20)

We use `generate_training_set` to perform the ASOF join to retrieve the features for the SPINEDATE's.

In [ ]:
training_set = fs.generate_training_set(
    spine_tbl,
    [fv_product_sales_lag_features_v1],
    spine_timestamp_col="SPINEDATE",
    join_method="cte",
)

print("Traning Set should be the same as the Spine record count: ", training_set.count()==spine_tbl.count())
print("Training Record Count: ", training_set.count()," Spine Count: ", spine_tbl.count())

Lets look at the Training Data we have retrieved, comparing the sparse versus the dense result.

In [ ]:
training_set.sort('SPINEDATE', 'PRODUCTKEY').show(100)

 Looking at the result we can see that for the missing dates in our sparse data (`PRODUCTKEY = 101`) we have different values returned versus the dense data (`PRODUCTKEY = 201`).  Compare the values for the `[2023-01-02, 2023-01-03, 2023-01-05, 2023-01-06, 2023-01-07]` dates for the 101 and 201 PRODUCTKEYs.

 Snowflake Feature Store constructs an ASOF join that looks back for the first available record by date that matches on the `PRODUCTKEY` join. Our LAG variables were derived relative to the `ORDERDATE`, not the `SPINEDATE`, and hence differ to those derived for the dense date, where we have every date available.

 We can see the constructed SQL from `generate_training_set()` below.

In [ ]:
print(training_set.queries['queries'][0])

This is most likely not the result we intended or wanted for these features. It's more likely that we want our lagged features to be derived relative to the `SPINEDATE`.  However, we only know the spine dates at query time, when we are retrieving training data for the dates we have a training label for, e.g. out-of-stock dates. 

One solution to this problem, as we have seen from our example data, is to generate a dense form of our event (`ORDER`) data before deriving the lagged variables, as per `PRODUCTKEY = 201`.  We have seen that this then returns the correct values relative to the Spine dates.  If the feature-data is dense, and our spine and feature data is using the same time-grain (e.g. DAY), the `ASOF` join SQL is not needed.  An `INNER JOIN` should suffice, as we have a feature record for every entity-key and timestamp. 

Snowflake provides several [gap-filling features and interpolation functions](https://docs.snowflake.com/en/user-guide/querying-time-series-data) that can be used for this purpose.  The disadvantage of the gap-filling approach is that our Feature View could become much larger (more rows) as a result.  In our test-data our dense data required 8 records, where the sparse form had three.  We could define the Feature View as a VIEW (`refresh_freq = None`), where the dense representation will be imputed at retrieval-time relative to the Spine, this avoids the need to pre-compute the full dense form, at the cost of deriving it for the scope of the spine for each retrieval operation. 

## Section 2

### Using Aggregation Feature Views - **New** 

> __Note:__ To run the code in this section you will need a version of the snowflake-ml package that is >= 1.24.0.

So we want a solution that does not require a dense interpolated form of the feature data to be pre-computed and stored, or derived from source data at retrieval time, once we have the Spine dates to work from.

The solution is provided by the new Time Aggregation Feature Views in Feature Store.  These provide new functionality to define temporal `features`, for a given time-grain (`feature_granularity`) within the definition of the Feature View.  When registered, the Feature View  creates a Dynamic Table or View that is an intermediate representation of the defined feature-values, that can be used more efficiently at query time to derive the final point-in-time accurate feature values.

Let's see what that looks like using our ORDERS source data.

First we define our features.

In [ ]:
sales_amount_lag_features = [ 
    Feature.sum(column="SALESAMOUNT", window = "1d", offset = "1d").alias("SALESAMOUNT_LAG1D"),        
    Feature.sum(column="SALESAMOUNT", window = "2d", offset = "1d").alias("SALESAMOUNT_LAG2D"),        
    Feature.sum(column="SALESAMOUNT", window = "3d", offset = "1d").alias("SALESAMOUNT_LAG3D"),        
    Feature.sum(column="SALESAMOUNT", window = "4d", offset = "1d").alias("SALESAMOUNT_LAG4D"),        
]


We can now use these within the Feature View definition

In [ ]:
fv_product_sales_lag_aggregation_features = FeatureView(
    name="FV_PRODUCT_SALES_LAG_AGGREGATION_FEATURES",
    entities=[ent_product],
    feature_df=order_tbl,                # Our Dataframe now references the raw Event Table
    timestamp_col="ORDERDATE",
    refresh_freq="1d",                   # How often the tiles are updated (e.g., every day)
    feature_granularity="1d",            # The size of each time tile (e.g., 1 day bucket)
    features=sales_amount_lag_features,  # Aggregate Feature definition
)

fv_product_sales_lag_aggregation_features_v1 = fs.register_feature_view(fv_product_sales_lag_aggregation_features, "V1", overwrite=True)

Once again, we use `generate_training_set` to perform the ASOF join to retrieve the features for the SPINEDATE's using the same Spine table we used earlier.

In [ ]:
training_set_aggregate = fs.generate_training_set(
    spine_tbl,
    [fv_product_sales_lag_aggregation_features_v1],
    spine_timestamp_col="SPINEDATE",
    include_feature_view_timestamp_col=True,  # This does not seem to be returning the Feature View's timestamp column when an Aggregation Feature View
    join_method="cte",
)
#.with_column_renamed(F.col("FV_PRODUCT_SALES_LAG_FEATURES_V1_ORDERDATE"), "ORDERDATE")

print("Traning Set should be the same as the Spine record count: ", training_set.count()==spine_tbl.count())
print("Training Record Count: ", training_set.count()," Spine Count: ", spine_tbl.count())

If we look at the training result for the new aggregated feature view, we can see that the values for PRODUCTKEY 101 and 201 now match, suggesting that we have resolved the issue when deriving temporal features over sparse time-series data.

In [ ]:
training_set_aggregate.sort('SPINEDATE',
#"ORDERDATE",
'PRODUCTKEY').show(100)

Let's compare the results from the Standard Feature View with those from the Aggregate Feature View by unioning them together. 

In [ ]:
std_agg_fv_results_combined_df = (training_set_aggregate
    .withColumn("FEATURE_VIEW", F.lit("AGG_FV")).union_all(
         training_set.drop('SALESAMOUNT')

          .withColumn("FEATURE_VIEW", F.lit("STD_FV"))
).select('FEATURE_VIEW','PRODUCTKEY','SPINEDATE',
         'SALESAMOUNT_LAG1D', 'SALESAMOUNT_LAG2D',
         'SALESAMOUNT_LAG3D', 'SALESAMOUNT_LAG4D')
.sort('PRODUCTKEY','SPINEDATE','FEATURE_VIEW')
)

Firstly, if we compare for PRODUCT_KEY 201 (dense time-series), where we know the results were correct in our standard Feature View.  The values for the Standard and Aggregated Feature Views should match ✅

In [ ]:
std_agg_fv_results_combined_df.filter(F.col('PRODUCTKEY')==F.lit(201)).show(50)

Now we look at the results for PRODUCT_KEY = 101.  We see that where ever we had time-series gaps [2023-01-02, 2023-01-03, 2023-01-05, 2023-01-06, 2023-01-07] in the time series for the Standard Feature View we have different values compared to the Aggregate Feature View.

The Aggregated Feature View is correctly deriving the feature-values relative to the Spine. ✅

In [ ]:
std_agg_fv_results_combined_df.filter(F.col('PRODUCTKEY')==F.lit(101)).show(50)

### Under the Hood
How does the new Aggregate Feature View work?  We will dig into a bit more detail on the internal implementation here for those that are curious, but the information below is not needed or necessary to use Aggregate Feature Views!

Instead of computing the final feature-values in the Feature View, we derive intermediate values that can be combined at retrieval time to derive the final point-in-time correct feature-value. 

Let's look at the Dynamic Table that was created from the Aggregate Feature View definition.

In [ ]:
session.table(f'{db_name}.FEATURE_STORE.FV_PRODUCT_SALES_LAG_AGGREGATION_FEATURES$V1').show()

This looks different to what we might have expected for the Feature View.  We are not storing the final SUM LAG columns in the Feature View.  Instead we have a _PARTIAL_SUM_SALESAMOUNT column. 

We have created a partial aggregation by truncating the source timestamp (`ORDER_DATE`) for the supplied time-grain (`feature_granularity="1d"`), and aggregating the input column (`column="SALESAMOUNT"`) over the entity-key and truncated date (`TILE_START`). This is often referred to as Aggregation Tiling in other Feature Stores.

Our aggregation features were defined thus:

```
sales_amount_lag_features = [ 
    Feature.sum(column="SALESAMOUNT", window = "1d", offset = "1d").alias("SALESAMOUNT_LAG1D"),        
    Feature.sum(column="SALESAMOUNT", window = "2d", offset = "1d").alias("SALESAMOUNT_LAG2D"),        
    Feature.sum(column="SALESAMOUNT", window = "3d", offset = "1d").alias("SALESAMOUNT_LAG3D"),        
    Feature.sum(column="SALESAMOUNT", window = "4d", offset = "1d").alias("SALESAMOUNT_LAG4D"),        
]
```

We only used a single column from our source table, and single aggregation type (`SUM`) in the feature definition. We have defined multiple lag-windows, but the same `_PARTIAL` column can be used at query time to derive all the lags in a single pass over the table using a conditional aggregation.  

Additional source columns and aggregation types would have resulted in additional `_PARTIAL` columns in the Feature View (Dynamic Table or View).

The Aggregation Feature View supports the following aggregation methods; SUM, COUNT, AVG, MIN, MAX, STD, VAR, FIRST_N, LAST_N, FIRST_DISTINCT_N, LAST_DISTINCT_N, APPROX_COUNT_DISTINCT, APPROX_PERCENTILE.

__Point-in-time Retrieval Query__

We can also inspect the generate SQL that derives the final point-in-time Feature values from the _PARTIAL feature-view. 

In [ ]:
print(training_set_aggregate.queries['queries'][0])

 We join the feature-view to the Spine via a single-pass over the feature-view. For each entity-key/spine-timestamp, we retrieve the relevant tiles (back to the max LAG window).  We use this to perform a conditional aggregation to derive each LAG window aggregation. e.g.

`SUM(CASE WHEN TILE_START >= DATEADD(DAY, -2, TILE_BOUNDARY) AND TILE_START < DATEADD(DAY, -1, TILE_BOUNDARY) THEN _PARTIAL_SUM_SALESAMOUNT ELSE 0 END) AS SALESAMOUNT_LAG1D,`

> __Note:__ We can see from this SQL that the larger the maximum LAG window required, the more data we will have to retrieve to derive the larger LAG windows.  Having seperate feature-views with coarser time `feature_granularity` can optimise and reduce the impact of very large LAG windows.  Likewise, lifetime (cummulative) features can be seperated into their own Feature Views.

In addition to the Feature View (Dynamic Table), an additional metadata table is created in the Feature Store schema, the first time an Aggregate Feature View is created in the Feature Store. This table stores the metadata related to the Feature View definition which is used by Feature Store to interpret the partial result table and derive the final feature SQL for it (**Look, but don't touch!** 
).

```
create or replace TABLE TIME_WINDOWED_FVIEWS.FEATURE_STORE._FEATURE_STORE_METADATA (
	OBJECT_TYPE VARCHAR(50) NOT NULL,
	OBJECT_NAME VARCHAR(256) NOT NULL,
	VERSION VARCHAR(128) NOT NULL,
	METADATA_TYPE VARCHAR(50) NOT NULL,
	METADATA VARIANT NOT NULL,
	CREATED_AT TIMESTAMP_NTZ(9) DEFAULT CURRENT_TIMESTAMP(),
	UPDATED_AT TIMESTAMP_NTZ(9) DEFAULT CURRENT_TIMESTAMP(),
	primary key (OBJECT_TYPE, OBJECT_NAME, VERSION, METADATA_TYPE)
) WITH TAG (MY_DB.FEATURE_STORE.SNOWML_FEATURE_STORE_OBJECT='{"type": "INTERNAL_METADATA_TABLE", "pkg_version": "1.23.0"}')
COMMENT='Internal metadata table for Feature Store. DO NOT modify directly - used for Feature Store internal operations.'
;
```

The table contains a pair of rows for every Aggregate Feature View.  If you modify the Aggregate Feature View, or overwite it, you may notice additional row-pairs, as per `UPDATE_AT` for the Feature View.

In [ ]:
session.table(f'{db_name}.FEATURE_STORE._FEATURE_STORE_METADATA').show()

### CAUTION: 
### Online Feature Tables

Aggregate Feature Views currently only work in an Offline (batch) context, and not for serving using the new Online Feature serving support in Feature Store (`online_config=OnlineConfig(enable=True)`). 
As we have seen the final-feature values are not stored in the Offline Feature View, only `_PARTIAL` results. 

If you were to enable an Aggregate Feature View for online-serving, only the latest (last) `TILE_START` value would be stored in the Online Table, rather than all the values for a given entity-key upto the greatest LAG window size which is needed to compute the final lagged feature values.  

Support for Online feature serving is under-development, and once available will support efficiently deriving and storing the correct latest feature-values for each of the defined LAG features in the Online Store.

----
## Author : Simon Field
----
## License

Copyright (c) 2025 Snowflake Inc.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.